In [1]:
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Working folder:", Path.cwd())

Python: C:\Users\USER\AppData\Local\Programs\Python\Python313\python.exe
Working folder: C:\Users\USER\google-geocoding-project


In [2]:
%pip install pandas openpyxl folium

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from pathlib import Path

data_path = (
    Path.cwd()
    / "data"
    / "processed"
    / "lagos_phc_geocoded.xlsx"
)

df = pd.read_excel(data_path)

print("Dataset shape:", df.shape)
print("Column names:", df.columns.tolist())

df.head()

Dataset shape: (325, 15)
Column names: ['S/N', 'LGA/LCDA', 'WARD', 'ADDRESS', 'District', 'PHC_NAME', 'google_address', 'place_id', 'lat', 'lon', 'geocode_method', 'status', 'query', 'google_name', 'partial_match']


,S/N,LGA/LCDA,WARD,ADDRESS,District,PHC_NAME,google_address,place_id,lat,lon,geocode_method,status,query,google_name,partial_match
0,1,ALIMOSHO LGA,ALABATA,"100 AKOWONJO ROAD,VULGANIDER B/S",District 1,AKOWONJO PHC,"106 Akowonjo Rd, Alimosho, Lagos 102213, Lagos...",ChIJHxPHb62ROxARx4cTFuB_VPM,6.600396,3.300375,NaN,FOUND,"AKOWONJO PHC, 100 AKOWONJO ROAD,VULGANIDER B/S...",Bola Tinubu Akowonjo Primary Health Centre,False
1,2,ALIMOSHO LGA,OGUNTADE BAMEKE,"20,EJIGBO ROAD,ORISUNMIBARE, SHASHA",District 1,ORISUNBARE PHC,"Orisunbare Rd, Isheri Olofin, Ikeja 102214, La...",ChIJCYqMIHGROxARq7n7xxRZmrw,6.570070,3.302892,NaN,FOUND,"ORISUNBARE PHC, 20,EJIGBO ROAD,ORISUNMIBARE, S...",Orisunbare Primary Health Care Centre,False
2,3,AGBADO OKE ODO LCDA,AGBADO OKEODO,"EKORO OKEODO ROAD, OKEODO ABULE EGBA",District 1,OKE ODO PHC,"J7QW+RFG, Old Otta Rd, Oke Odo, Lagos 102213, ...",ChIJTVGehnaROxARtrDYIEsQ5m8,6.639574,3.296210,NaN,FOUND,"OKE ODO PHC, EKORO OKEODO ROAD, OKEODO ABULE E...",Oke-Odo Primary Healthcare Centre,False
3,4,AGBADO OKE ODO LCDA,AJASA/AMIKANLE,"3, FAITH RESCUE BY COMMAND SECONDARY SCHOOL",District 1,SURULERE PHC.,"3 Akerele St, Akerele Extension, Lagos 101241,...",ChIJyVt6ZD6MOxARFLnPAsGPgt8,6.506216,3.352641,NaN,FOUND,"SURULERE PHC., 3, FAITH RESCUE BY COMMAND SECO...",Primary Healthcare Akerele,False
4,5,AGBADO OKE ODO LCDA,AJASA/AMIKANLE,"2/5, FADIPE STR, AGBADO",District 1,AMIKANLE PHC,"12 Peter Odeyale St, Oke Odo, Lagos 102213, La...",ChIJEXRPsleROxARMAQg-XIcxas,6.638108,3.284097,NaN,FOUND,"AMIKANLE PHC, 2/5, FADIPE STR, AGBADO, AGBADO ...",AGBELEKALE PRIMARY HEALTH CARE CENTRE,False


In [2]:
# Work on a copy, keeping the loaded dataset unchanged
map_df = df.copy()

# Convert coordinates to numbers
map_df["lat"] = pd.to_numeric(map_df["lat"], errors="coerce")
map_df["lon"] = pd.to_numeric(map_df["lon"], errors="coerce")

# Check for missing or out-of-range coordinates
valid_coordinates = (
    map_df["lat"].between(-90, 90)
    & map_df["lon"].between(-180, 180)
)

print("Total PHC records:", len(map_df))
print("Usable coordinates:", valid_coordinates.sum())
print("Missing or invalid coordinates:", (~valid_coordinates).sum())

# Flag records sharing exactly the same location
map_df["shared_coordinates"] = map_df.duplicated(
    subset=["lat", "lon"],
    keep=False
) & valid_coordinates

print("Records sharing coordinates:", map_df["shared_coordinates"].sum())

Total PHC records: 325
Usable coordinates: 325
Missing or invalid coordinates: 0
Records sharing coordinates: 133


In [3]:
import folium

# Create a map centred around Lagos
phc_map = folium.Map(
    location=[6.55, 3.40],
    zoom_start=10,
    tiles="OpenStreetMap",
    control_scale=True
)

# Display the map in JupyterLab
phc_map

In [6]:
import folium

phc_map = folium.Map(
    location=[6.55, 3.40],
    zoom_start=10,
    tiles="OpenStreetMap",
    control_scale=True,
    height=550
)

# Find the southwest and northeast corners of the PHC locations
bounds = [
    [map_df["lat"].min(), map_df["lon"].min()],
    [map_df["lat"].max(), map_df["lon"].max()]
]

# Adjust the map view to include all PHC locations
phc_map.fit_bounds(bounds, padding=(20, 20))

phc_map

In [5]:
from folium.plugins import MarkerCluster
from html import escape

# Group nearby markers and allow overlapping markers to spread out
phc_cluster = MarkerCluster(
    name="Primary Health Centres",
    spiderfyOnMaxZoom=True
).add_to(phc_map)

# Add one marker for each PHC
for _, row in map_df.iterrows():

    hover_info = (
        f"<b>{escape(str(row['PHC_NAME']))}</b><br>"
        f"LGA/LCDA: {escape(str(row['LGA/LCDA']))}<br>"
        f"Ward: {escape(str(row['WARD']))}"
    )

    folium.Marker(
        location=[row["lat"], row["lon"]],
        tooltip=folium.Tooltip(hover_info),
        icon=folium.Icon(color="blue", icon="plus-sign")
    ).add_to(phc_cluster)

phc_map

In [7]:
from folium.plugins import MarkerCluster
from html import escape

# Prepare text for display, including missing values
def display_text(value):
    if pd.isna(value):
        return "Not recorded"
    return escape(str(value))


phc_cluster = MarkerCluster(
    name="Primary Health Centres",
    spiderfyOnMaxZoom=True
).add_to(phc_map)


for _, row in map_df.iterrows():

    hover_info = (
        f"<b>{display_text(row['PHC_NAME'])}</b><br>"
        f"LGA/LCDA: {display_text(row['LGA/LCDA'])}<br>"
        f"Ward: {display_text(row['WARD'])}"
    )

    location_note = (
        "Shares coordinates with another record; review needed."
        if row["shared_coordinates"]
        else "No shared coordinates detected."
    )

    popup_info = f"""
    <div style="font-family: Arial; font-size: 13px; width: 270px;">
        <h4>{display_text(row['PHC_NAME'])}</h4>

        <b>LGA/LCDA:</b> {display_text(row['LGA/LCDA'])}<br>
        <b>Ward:</b> {display_text(row['WARD'])}<br>
        <b>District:</b> {display_text(row['District'])}<br><br>

        <b>Original address:</b><br>
        {display_text(row['ADDRESS'])}<br><br>

        <b>Google facility name:</b><br>
        {display_text(row['google_name'])}<br><br>

        <b>Google address:</b><br>
        {display_text(row['google_address'])}<br><br>

        <b>Coordinates:</b>
        {row['lat']:.6f}, {row['lon']:.6f}<br><br>

        <b>Location check:</b> {location_note}<br>
        <small>Geocoded location; accuracy has not been
        independently confirmed for every record.</small>
    </div>
    """

    folium.Marker(
        location=[row["lat"], row["lon"]],
        tooltip=folium.Tooltip(hover_info),
        popup=folium.Popup(popup_info, max_width=320),
        icon=folium.Icon(color="blue", icon="plus-sign")
    ).add_to(phc_cluster)

phc_map

In [8]:
from folium.plugins import Fullscreen

Fullscreen(
    position="topleft",
    title="Open fullscreen",
    title_cancel="Exit fullscreen"
).add_to(phc_map)

phc_map

In [10]:
from branca.element import Element

map_header = f"""
<div style="
    position: fixed;
    top: 12px;
    left: 60px;
    z-index: 1000;
    background: white;
    padding: 12px 16px;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.25);
    max-width: min(340px, calc(100vw - 100px));
    font-family: Arial;
">
    <div style="font-size: 17px; font-weight: bold;">
        Lagos Primary Health Centres
    </div>

    <div style="font-size: 13px; margin-top: 5px;">
        Exploring {len(map_df)} geocoded PHC records
    </div>

    <details style="font-size: 12px; margin-top: 8px;">
        <summary style="cursor: pointer;">About this map</summary>

        <p>Hover over a marker for a summary.
        Click or tap it for full details.
        Click numbered clusters to explore nearby records.</p>

        <p>Source: Lagos State Ministry of Health facility list.
        Locations obtained through Google Places and Geocoding APIs.</p>

        <p>Some records share coordinates and require review.
        Locations have not all been independently verified.</p>

        <p style="margin-bottom: 0;">
            Created by Ashogbon Adewale Emmanuel
        </p>
    </details>
</div>
"""

phc_map.get_root().html.add_child(
    Element(map_header),
    name="map_header"
)

phc_map

In [11]:
# Locate the project using the dataset path
project_folder = data_path.parent.parent.parent

# Create the docs folder if needed
output_folder = project_folder / "docs"
output_folder.mkdir(parents=True, exist_ok=True)

# Save the interactive map
map_file = output_folder / "phc_map.html"
phc_map.save(str(map_file))

print("Map saved to:", map_file)

Map saved to: C:\Users\USER\google-geocoding-project\docs\phc_map.html
